# EDA — Acidentes em Rodovias Federais Brasileiras (2007–2020)

**Disciplina:** Análise e Visualização de Dados  
**Dataset:** [Kaggle — Acidentes em Rodovias Federais (2007–2020)](https://www.kaggle.com/datasets/marcosfnlr/acidentes-em-rodovias-federais-20072020/data)

## Objetivo
Realizar uma **Análise Exploratória de Dados (EDA)** completa, investigando qualidade dos dados, distribuições, valores ausentes, outliers e relações entre variáveis, sempre formulando perguntas e discutindo os resultados.

### Perguntas que guiam a análise
1. Como o número de acidentes e de vítimas fatais evoluiu entre 2007 e 2020?
2. Quais estados e rodovias concentram mais acidentes? A concentração de acidentes é igual à concentração de fatalidades?
3. Em quais dias da semana e fases do dia há mais ocorrências?
4. Quais causas e tipos de acidente aparecem com maior frequência e quais estão mais associados a fatalidades?
5. Condição meteorológica, tipo de pista e traçado da via apresentam diferenças na proporção de acidentes fatais?
6. Existe relação entre quantidade de veículos, pessoas, feridos e mortos em uma ocorrência?

> **Cuidado de interpretação:** este dataset contém acidentes registrados, e não a exposição ao risco (fluxo de veículos, quilômetros percorridos, extensão de cada BR etc.). Portanto, uma categoria ter mais acidentes **não significa necessariamente** que ela seja mais perigosa.

## 1. Conhecimento de domínio e fonte dos dados

Os dados representam ocorrências registradas em **rodovias federais brasileiras**. Entre as variáveis estão data, dia da semana, UF, BR, município, causa e tipo do acidente, classificação de gravidade, condições meteorológicas, tipo/traçado da pista e totais de pessoas, mortos, feridos e veículos envolvidos.

A fonte original é a **Polícia Rodoviária Federal (PRF)**, posteriormente organizada e disponibilizada no Kaggle. Como o período é longo (2007–2020), é plausível que existam mudanças de preenchimento, padronização e disponibilidade de campos ao longo dos anos. Isso será considerado principalmente na análise de dados ausentes.

Fontes de apoio:
- Dataset: https://www.kaggle.com/datasets/marcosfnlr/acidentes-em-rodovias-federais-20072020/data
- Dicionário PRF (variáveis de acidentes): https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dicionario-dados-acidentes-2016

## 2. Preparação do ambiente e carregamento dos dados

In [ ]:
# Instala apenas se necessário. No Google Colab, kagglehub permite baixar datasets públicos.
!pip -q install kagglehub

In [ ]:
import os
import glob
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
COR_PRINCIPAL = '#4C78A8'
COR_DESTAQUE = '#E45756'

In [ ]:
# 1) Se o arquivo já estiver no ambiente, usa o arquivo local.
# 2) Caso contrário, baixa automaticamente do Kaggle.

arquivos_locais = []
for pasta in ['.', '/content', '/mnt/data']:
    arquivos_locais.extend(glob.glob(os.path.join(pasta, '**', '*.csv'), recursive=True))

preferidos = [p for p in arquivos_locais if 'acidentes2007-2020' in os.path.basename(p).lower()]

if preferidos:
    csv_path = preferidos[0]
    print('Arquivo local encontrado:', csv_path)
else:
    try:
        import kagglehub
        dataset_dir = kagglehub.dataset_download('marcosfnlr/acidentes-em-rodovias-federais-20072020')
        csvs = glob.glob(os.path.join(dataset_dir, '**', '*.csv'), recursive=True)
        if not csvs:
            raise FileNotFoundError('Nenhum CSV foi encontrado no dataset baixado.')
        preferidos = [p for p in csvs if 'acidentes2007-2020' in os.path.basename(p).lower()]
        csv_path = preferidos[0] if preferidos else max(csvs, key=os.path.getsize)
        print('Arquivo baixado do Kaggle:', csv_path)
    except Exception as erro_kaggle:
        print('Download automático não funcionou:', erro_kaggle)
        print('Faça upload do CSV acidentes2007-2020.csv quando a janela aparecer.')
        try:
            from google.colab import files
            enviados = files.upload()
            csvs = [nome for nome in enviados if nome.lower().endswith('.csv')]
            if not csvs:
                raise FileNotFoundError('Nenhum CSV foi enviado.')
            csv_path = csvs[0]
        except ImportError:
            raise RuntimeError('Execute no Colab ou coloque acidentes2007-2020.csv na mesma pasta do notebook.')

# Tenta os encodings mais comuns
ultimo_erro = None
for encoding in ['utf-8', 'utf-8-sig', 'latin1']:
    try:
        df = pd.read_csv(csv_path, low_memory=False, encoding=encoding)
        print(f'Leitura concluída com encoding={encoding}')
        break
    except Exception as e:
        ultimo_erro = e
else:
    raise ultimo_erro

print(f'Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}')
df.head()

### 2.1 Inspeção inicial

Nesta etapa verificamos tamanho, nomes das colunas, tipos, duplicidades e exemplos de registros. Antes de alterar qualquer coisa, guardamos uma cópia dos dados brutos.

In [ ]:
linhas_brutas, colunas_brutas = df.shape

print('DIMENSÃO:', df.shape)
print('\nCOLUNAS:')
print(df.columns.tolist())
print('\nTIPOS:')
display(df.dtypes.to_frame('dtype'))
print('\nDUPLICATAS EXATAS:', df.duplicated().sum())

In [ ]:
# Função para localizar colunas mesmo quando existem pequenas variações de nome.
def primeira_coluna(*opcoes):
    mapa = {c.lower().strip(): c for c in df.columns}
    for opcao in opcoes:
        if opcao.lower() in mapa:
            return mapa[opcao.lower()]
    return None

C_DATA = primeira_coluna('data_inversa', 'data')
C_ID = primeira_coluna('id')
C_ID_UNICO = primeira_coluna('id_unico')
C_DIA = primeira_coluna('dia_semana')
C_HORA = primeira_coluna('horario')
C_UF = primeira_coluna('uf')
C_BR = primeira_coluna('br')
C_KM = primeira_coluna('km')
C_MUNICIPIO = primeira_coluna('municipio')
C_CAUSA = primeira_coluna('causa_acidente')
C_TIPO = primeira_coluna('tipo_acidente')
C_CLASS = primeira_coluna('classificacao_acidente', 'classificação_acidente')
C_FASE = primeira_coluna('fase_dia')
C_CLIMA = primeira_coluna('condicao_metereologica', 'condicao_meteorologica', 'condição_meteorologica')
C_PISTA = primeira_coluna('tipo_pista')
C_TRACADO = primeira_coluna('tracado_via', 'traçado_via')
C_USO_SOLO = primeira_coluna('uso_solo')
C_PESSOAS = primeira_coluna('pessoas')
C_MORTOS = primeira_coluna('mortos')
C_FERIDOS_LEVES = primeira_coluna('feridos_leves')
C_FERIDOS_GRAVES = primeira_coluna('feridos_graves')
C_ILESOS = primeira_coluna('ilesos')
C_IGNORADOS = primeira_coluna('ignorados')
C_FERIDOS = primeira_coluna('feridos')
C_VEICULOS = primeira_coluna('veiculos', 'veículos')

cols_importantes = {
    'data': C_DATA, 'dia_semana': C_DIA, 'horario': C_HORA, 'uf': C_UF, 'br': C_BR,
    'km': C_KM, 'municipio': C_MUNICIPIO, 'causa': C_CAUSA, 'tipo': C_TIPO,
    'classificacao': C_CLASS, 'fase_dia': C_FASE, 'clima': C_CLIMA, 'tipo_pista': C_PISTA,
    'tracado_via': C_TRACADO, 'uso_solo': C_USO_SOLO, 'pessoas': C_PESSOAS,
    'mortos': C_MORTOS, 'feridos_leves': C_FERIDOS_LEVES, 'feridos_graves': C_FERIDOS_GRAVES,
    'ilesos': C_ILESOS, 'ignorados': C_IGNORADOS, 'feridos': C_FERIDOS, 'veiculos': C_VEICULOS
}

display(pd.Series(cols_importantes, name='coluna_encontrada').to_frame())

## 3. Limpeza mínima e criação de variáveis auxiliares

In [ ]:
# Padroniza strings vazias e marcadores comuns de ausência.
marcadores_nulos = ['(null)', 'null', 'NULL', 'NA', 'N/A', 'Ignorado', 'Não Informado', '']

cols_texto = [c for c in [C_DIA, C_UF, C_MUNICIPIO, C_CAUSA, C_TIPO, C_CLASS, C_FASE, C_CLIMA, C_PISTA, C_TRACADO, C_USO_SOLO, C_ID_UNICO] if c]
for col in cols_texto:
    df[col] = df[col].astype('string').str.strip()
    df[col] = df[col].replace(marcadores_nulos, pd.NA)

# Converte colunas de contagem para numérico quando disponíveis.
cols_contagem = [c for c in [C_PESSOAS, C_MORTOS, C_FERIDOS_LEVES, C_FERIDOS_GRAVES,
                              C_ILESOS, C_IGNORADOS, C_FERIDOS, C_VEICULOS] if c]
for col in cols_contagem:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# BR e KM podem vir como string/float; convertemos para auxiliar análises geográficas.
if C_BR:
    df[C_BR] = pd.to_numeric(df[C_BR], errors='coerce')
if C_KM:
    df[C_KM] = pd.to_numeric(df[C_KM].astype('string').str.replace(',', '.', regex=False), errors='coerce')

# Criação robusta da data.
df['data'] = pd.NaT
if C_DATA:
    df['data'] = pd.to_datetime(df[C_DATA], errors='coerce', dayfirst=True)

# Em versões deste dataset, id_unico contém a data no final. Usamos como fallback.
if C_ID_UNICO:
    texto = df[C_ID_UNICO].astype('string')
    extraida = texto.str.extract(r'(\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4})', expand=False)
    data_fallback = pd.to_datetime(extraida, errors='coerce', dayfirst=True)
    df['data'] = df['data'].fillna(data_fallback)

df['ano'] = df['data'].dt.year
df['mes'] = df['data'].dt.month

# Horário e hora inteira
if C_HORA:
    hora_temp = pd.to_datetime(df[C_HORA].astype('string'), format='%H:%M:%S', errors='coerce')
    df['hora'] = hora_temp.dt.hour

# Indicador de acidente fatal. Mortos > 0 é a definição principal quando a coluna existe.
if C_MORTOS:
    df['fatal'] = df[C_MORTOS].fillna(0) > 0
elif C_CLASS:
    df['fatal'] = df[C_CLASS].astype('string').str.contains('Fata', case=False, na=False)
else:
    df['fatal'] = False

# Taxas internas da ocorrência — úteis para descrição, não representam risco populacional.
if C_PESSOAS and C_MORTOS:
    df['proporcao_mortos_envolvidos'] = np.where(df[C_PESSOAS] > 0, df[C_MORTOS] / df[C_PESSOAS], np.nan)

print('Período identificado:', df['ano'].min(), 'a', df['ano'].max())
print('Acidentes fatais:', int(df['fatal'].sum()), f"({df['fatal'].mean()*100:.2f}%)")

### Discussão da limpeza

- Colunas como `br` são identificadores numéricos, portanto **não devem ser interpretadas como uma variável quantitativa contínua**.
- `km` representa posição na rodovia; pode ser numérico, mas uma média de quilômetros entre BRs diferentes tem pouco sentido substantivo.
- As variáveis mais adequadas para distribuição quantitativa são as contagens de pessoas, veículos, mortos e feridos.
- Não removemos linhas em massa nesta etapa. A estratégia será tratar ausências **por análise**, evitando descartar registros úteis por causa de um campo irrelevante para aquela pergunta.

## 4. Dados ausentes

In [ ]:
missing = pd.DataFrame({
    'ausentes': df.isna().sum(),
    'percentual': df.isna().mean().mul(100)
}).sort_values('percentual', ascending=False)

display(missing.head(20).round(2))

plt.figure(figsize=(10, 6))
top_missing = missing[missing['ausentes'] > 0].head(15).sort_values('percentual')
if not top_missing.empty:
    plt.barh(top_missing.index, top_missing['percentual'])
    plt.xlabel('% de valores ausentes')
    plt.title('Top 15 colunas com maior percentual de dados ausentes')
    plt.tight_layout()
    plt.show()
else:
    print('Não foram identificados valores ausentes após a padronização.')

### As ausências ocorrem juntas?

Só contar nulos por coluna não mostra se diferentes campos ficam ausentes nas **mesmas linhas**. Como a base é muito grande, usamos uma amostra para visualizar a associação entre indicadores de ausência. Em seguida verificamos se a ausência varia por ano, o que ajuda a detectar possíveis mudanças de processo de coleta.

In [ ]:
cols_com_ausentes = missing.query('ausentes > 0').head(10).index.tolist()

if len(cols_com_ausentes) >= 2:
    amostra_missing = df[cols_com_ausentes].sample(min(50000, len(df)), random_state=RANDOM_STATE)
    corr_missing = amostra_missing.isna().astype(int).corr()
    plt.figure(figsize=(9, 7))
    sns.heatmap(corr_missing, cmap='coolwarm', center=0, vmin=-1, vmax=1)
    plt.title('Correlação entre indicadores de ausência (amostra)')
    plt.tight_layout()
    plt.show()
else:
    print('Poucas colunas com ausência para construir correlação de missingness.')

In [ ]:
# Missingness por ano: se a taxa muda muito, a ausência provavelmente não é completamente aleatória.
if df['ano'].notna().any() and cols_com_ausentes:
    cols_miss_ano = cols_com_ausentes[:8]
    miss_ano = df.groupby('ano')[cols_miss_ano].agg(lambda s: s.isna().mean() * 100)
    display(miss_ano.round(2))

    plt.figure(figsize=(12, 6))
    for col in cols_miss_ano:
        plt.plot(miss_ano.index, miss_ano[col], marker='o', label=col)
    plt.ylabel('% ausente')
    plt.title('Percentual de ausência por ano')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
# Discussão automática sobre mecanismo de ausência
if cols_com_ausentes and df['ano'].notna().any():
    variacao = {}
    for col in cols_com_ausentes[:8]:
        taxas = df.groupby('ano')[col].apply(lambda s: s.isna().mean())
        variacao[col] = taxas.max() - taxas.min()
    maior_col = max(variacao, key=variacao.get)
    print('DISCUSSÃO:')
    print(f"A coluna '{maior_col}' apresenta uma variação de aproximadamente {variacao[maior_col]*100:.1f} pontos percentuais de ausência entre anos.")
    print('Isso sugere que pelo menos parte da ausência pode estar ligada ao período/protocolo de coleta (compatível com MAR), e não ser puramente MCAR.')
    print('Não é possível concluir MNAR apenas observando esta base. Por isso, o tratamento será conservador: não imputar categorias arbitrariamente e remover nulos apenas nas análises que exigem aquele campo.')

## 5. Princípio distribucional — variáveis quantitativas

In [ ]:
VARS_QUANT = [c for c in [C_PESSOAS, C_MORTOS, C_FERIDOS_LEVES, C_FERIDOS_GRAVES,
                           C_ILESOS, C_IGNORADOS, C_FERIDOS, C_VEICULOS] if c]

print('Variáveis quantitativas usadas:', VARS_QUANT)

desc = df[VARS_QUANT].describe(percentiles=[.25, .5, .75]).T
desc['skewness'] = df[VARS_QUANT].skew(numeric_only=True)
desc['kurtosis'] = df[VARS_QUANT].kurtosis(numeric_only=True)
desc['IQR'] = desc['75%'] - desc['25%']

display(desc[['count','mean','50%','std','min','25%','75%','max','IQR','skewness','kurtosis']].rename(columns={'50%':'median'}).round(2))

In [ ]:
# Histograma em amostra para não sobrecarregar a visualização de 1,8M+ registros.
amostra_plot = df.sample(min(100000, len(df)), random_state=RANDOM_STATE)

n = len(VARS_QUANT)
fig, axes = plt.subplots(nrows=(n + 2)//3, ncols=3, figsize=(16, 4*((n+2)//3)))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, VARS_QUANT):
    sns.histplot(amostra_plot[col].dropna(), bins=40, kde=True, color=COR_PRINCIPAL, ax=ax)
    ax.set_title(f'Distribuição — {col}')

for ax in axes[len(VARS_QUANT):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

### Teste de normalidade

O **Shapiro-Wilk** é muito sensível em amostras enormes. Para evitar que milhões de observações tornem qualquer desvio trivial “significativo”, testamos uma amostra aleatória de até 5.000 valores por variável e combinamos o p-valor com histogramas, assimetria e curtose.

In [ ]:
resultados_shapiro = []
for col in VARS_QUANT:
    valores = df[col].dropna()
    if len(valores) >= 3:
        amostra = valores.sample(min(5000, len(valores)), random_state=RANDOM_STATE)
        stat, p = stats.shapiro(amostra)
        resultados_shapiro.append({
            'variavel': col,
            'n_teste': len(amostra),
            'estatistica': stat,
            'p_valor': p,
            'rejeita_normalidade_5%': p < 0.05
        })

shapiro_df = pd.DataFrame(resultados_shapiro).set_index('variavel')
display(shapiro_df.round(6))

In [ ]:
# QQ-Plot de uma variável representativa: pessoas ou veículos.
qq_col = C_PESSOAS if C_PESSOAS else (C_VEICULOS if C_VEICULOS else VARS_QUANT[0])
valores = df[qq_col].dropna().sample(min(5000, df[qq_col].notna().sum()), random_state=RANDOM_STATE)
plt.figure(figsize=(6,6))
stats.probplot(valores, dist='norm', plot=plt)
plt.title(f'QQ-Plot — {qq_col}')
plt.tight_layout()
plt.show()

### Interpretação distribucional

As variáveis deste dataset são **contagens discretas** e costumam ter muitos valores baixos (0, 1, 2...) e uma cauda longa formada por acidentes de maior porte. Assim, normalidade não é uma expectativa natural. Isso justifica priorizar **mediana/IQR**, correlação de **Spearman** e testes não paramétricos em comparações de grupos.

## 6. Outliers — detecção por IQR e decisão de tratamento

In [ ]:
def resumo_iqr(serie):
    s = serie.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    n_out = ((s < lim_inf) | (s > lim_sup)).sum()
    return pd.Series({
        'Q1': q1, 'Q3': q3, 'IQR': iqr,
        'limite_inferior': lim_inf, 'limite_superior': lim_sup,
        'n_outliers_IQR': n_out, 'pct_outliers': n_out/len(s)*100 if len(s) else np.nan,
        'max': s.max() if len(s) else np.nan
    })

outliers_resumo = pd.DataFrame({col: resumo_iqr(df[col]) for col in VARS_QUANT}).T
display(outliers_resumo.round(2))

In [ ]:
fig, axes = plt.subplots(nrows=(len(VARS_QUANT)+3)//4, ncols=4, figsize=(16, 4*((len(VARS_QUANT)+3)//4)))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, VARS_QUANT):
    sns.boxplot(y=amostra_plot[col], color=COR_PRINCIPAL, ax=ax)
    ax.set_title(col)
for ax in axes[len(VARS_QUANT):]:
    ax.axis('off')
plt.suptitle('Boxplots — identificação visual de valores extremos', y=1.01)
plt.tight_layout()
plt.show()

### Decisão sobre outliers

Não removeremos automaticamente os outliers. Em dados de acidentes, ocorrências envolvendo muitos veículos/pessoas podem ser **raras, mas legítimas** (engavetamentos, ônibus, colisões múltiplas). Excluir esses casos poderia apagar justamente os eventos mais graves e informativos.

A regra do IQR é usada aqui como **sinalizador**, não como regra automática de exclusão. Valores extremos só deveriam ser removidos após comprovação de erro de registro ou impossibilidade física.

## 7. Variáveis qualitativas — frequência e desbalanceamento

In [ ]:
VARS_QUALI = [c for c in [C_DIA, C_UF, C_CAUSA, C_TIPO, C_CLASS, C_FASE, C_CLIMA, C_PISTA, C_TRACADO, C_USO_SOLO] if c]

resumo_quali = []
for col in VARS_QUALI:
    vc = df[col].value_counts(dropna=True)
    resumo_quali.append({
        'variavel': col,
        'n_categorias': vc.size,
        'moda': vc.index[0] if len(vc) else np.nan,
        'freq_moda': vc.iloc[0] if len(vc) else np.nan,
        'pct_moda': (vc.iloc[0]/vc.sum()*100) if len(vc) else np.nan
    })

display(pd.DataFrame(resumo_quali).set_index('variavel').round(2))

In [ ]:
# Mostra as 10 categorias mais frequentes das principais qualitativas.
principais_quali = [c for c in [C_UF, C_CAUSA, C_TIPO, C_CLASS, C_FASE, C_CLIMA, C_PISTA, C_TRACADO] if c]

for col in principais_quali:
    top = df[col].value_counts().head(10).sort_values()
    plt.figure(figsize=(9, 5))
    plt.barh(top.index.astype(str), top.values)
    plt.title(f'Top categorias — {col}')
    plt.xlabel('Quantidade de acidentes')
    plt.tight_layout()
    plt.show()

## 8. Pergunta 1 — Evolução temporal dos acidentes e das fatalidades

In [ ]:
if df['ano'].notna().any():
    temporal = df.groupby('ano').agg(
        acidentes=('fatal', 'size'),
        acidentes_fatais=('fatal', 'sum')
    )
    if C_MORTOS:
        temporal['mortos'] = df.groupby('ano')[C_MORTOS].sum(min_count=1)
    temporal['pct_acidentes_fatais'] = temporal['acidentes_fatais'] / temporal['acidentes'] * 100
    display(temporal.round(2))

    plt.figure(figsize=(11,5))
    plt.plot(temporal.index, temporal['acidentes'], marker='o')
    plt.title('Evolução anual do número de acidentes')
    plt.xlabel('Ano')
    plt.ylabel('Acidentes')
    plt.xticks(temporal.index, rotation=45)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(11,5))
    plt.plot(temporal.index, temporal['pct_acidentes_fatais'], marker='o')
    plt.title('Percentual de acidentes fatais por ano')
    plt.xlabel('Ano')
    plt.ylabel('% de acidentes fatais')
    plt.xticks(temporal.index, rotation=45)
    plt.tight_layout()
    plt.show()

    ano_pico = temporal['acidentes'].idxmax()
    ano_min = temporal['acidentes'].idxmin()
    queda = (1 - temporal.loc[ano_min,'acidentes']/temporal.loc[ano_pico,'acidentes']) * 100
    print('DISCUSSÃO:')
    print(f'O maior volume anual foi em {int(ano_pico)}, com {int(temporal.loc[ano_pico,"acidentes"]):,} acidentes.')
    print(f'O menor volume da série foi em {int(ano_min)}, com {int(temporal.loc[ano_min,"acidentes"]):,} acidentes.')
    print(f'Entre o pico e o menor valor observado, a redução foi de aproximadamente {queda:.1f}%.')
    if 2020 in temporal.index:
        print('O ano de 2020 deve ser interpretado com cautela, pois mudanças de mobilidade durante a pandemia podem ter afetado o volume de tráfego e registros.')

## 9. Pergunta 2 — Dias da semana e fases do dia

In [ ]:
if C_DIA:
    # Normalização simples para ordenar nomes comuns em português.
    ordem_dias = ['segunda-feira','terça-feira','quarta-feira','quinta-feira','sexta-feira','sábado','domingo']
    serie_dia = df[C_DIA].astype('string').str.lower().str.strip()
    cont_dia = serie_dia.value_counts()
    ordem_existente = [d for d in ordem_dias if d in cont_dia.index]
    restantes = [d for d in cont_dia.index if d not in ordem_existente]
    ordem = ordem_existente + restantes

    plt.figure(figsize=(10,5))
    sns.countplot(x=serie_dia, order=ordem, color=COR_PRINCIPAL)
    plt.title('Acidentes por dia da semana')
    plt.xlabel('Dia da semana')
    plt.ylabel('Acidentes')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

    dia_top = cont_dia.index[0]
    print(f'DISCUSSÃO: o dia com maior número de registros é {dia_top}, com {cont_dia.iloc[0]:,} acidentes.')

if C_FASE:
    fase = df[C_FASE].value_counts()
    plt.figure(figsize=(8,5))
    sns.barplot(x=fase.values, y=fase.index, color=COR_PRINCIPAL)
    plt.title('Acidentes por fase do dia')
    plt.xlabel('Acidentes')
    plt.ylabel('Fase do dia')
    plt.tight_layout()
    plt.show()

    taxa_fase = df.groupby(C_FASE)['fatal'].agg(['count','mean']).sort_values('mean', ascending=False)
    taxa_fase['pct_fatais'] = taxa_fase['mean']*100
    display(taxa_fase[['count','pct_fatais']].round(2))
    print('DISCUSSÃO: a fase com mais acidentes não precisa ser a mesma com maior proporção de acidentes fatais. Isso separa volume de ocorrência e gravidade.')

In [ ]:
if 'hora' in df.columns:
    por_hora = df.groupby('hora')['fatal'].agg(['count','mean'])
    por_hora['pct_fatais'] = por_hora['mean']*100

    fig, ax1 = plt.subplots(figsize=(11,5))
    ax1.bar(por_hora.index, por_hora['count'], alpha=.7)
    ax1.set_xlabel('Hora do dia')
    ax1.set_ylabel('Número de acidentes')
    ax1.set_title('Acidentes por hora do dia')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(11,5))
    plt.plot(por_hora.index, por_hora['pct_fatais'], marker='o')
    plt.xlabel('Hora do dia')
    plt.ylabel('% de acidentes fatais')
    plt.title('Proporção de acidentes fatais por hora')
    plt.tight_layout()
    plt.show()

## 10. Pergunta 3 — Estados e rodovias que concentram ocorrências

In [ ]:
if C_UF:
    por_uf = df.groupby(C_UF)['fatal'].agg(acidentes='size', fatais='sum')
    por_uf['pct_fatais'] = por_uf['fatais']/por_uf['acidentes']*100
    por_uf = por_uf.sort_values('acidentes', ascending=False)
    display(por_uf.head(15).round(2))

    plt.figure(figsize=(11,6))
    top_ufs = por_uf.head(15).sort_values('acidentes')
    plt.barh(top_ufs.index, top_ufs['acidentes'])
    plt.title('Top 15 UFs por número de acidentes')
    plt.xlabel('Acidentes')
    plt.tight_layout()
    plt.show()

    # Taxa de fatalidade apenas em UFs com volume relevante
    relevantes = por_uf[por_uf['acidentes'] >= por_uf['acidentes'].quantile(.25)].sort_values('pct_fatais', ascending=False)
    print('UFs com maior proporção de acidentes fatais (entre UFs com volume relevante):')
    display(relevantes.head(10).round(2))
    print(f"DISCUSSÃO: {por_uf.index[0]} concentra o maior número absoluto de acidentes ({por_uf.iloc[0]['acidentes']:,.0f}).")
    print('Isso não prova maior risco rodoviário, pois não temos fluxo de veículos, extensão de rodovias nem quilômetros percorridos por UF.')

In [ ]:
if C_BR:
    df_br = df.dropna(subset=[C_BR]).copy()
    df_br['BR'] = df_br[C_BR].round().astype('Int64').astype('string')
    por_br = df_br.groupby('BR')['fatal'].agg(acidentes='size', fatais='sum')
    por_br['pct_fatais'] = por_br['fatais']/por_br['acidentes']*100
    por_br = por_br.sort_values('acidentes', ascending=False)
    display(por_br.head(15).round(2))

    top_br = por_br.head(15).sort_values('acidentes')
    plt.figure(figsize=(10,6))
    plt.barh(['BR-' + x for x in top_br.index.astype(str)], top_br['acidentes'])
    plt.title('Top 15 rodovias por número de acidentes')
    plt.xlabel('Acidentes')
    plt.tight_layout()
    plt.show()

    print(f"DISCUSSÃO: a BR-{por_br.index[0]} aparece com o maior volume absoluto na base.")
    print('Novamente, volume não equivale a risco: rodovias mais longas e movimentadas tendem a acumular mais eventos.')

## 11. Pergunta 4 — Causas e tipos de acidente: frequência x gravidade

In [ ]:
def resumo_categoria_fatal(col, minimo=500):
    tab = df.groupby(col)['fatal'].agg(acidentes='size', fatais='sum')
    tab['pct_fatais'] = tab['fatais']/tab['acidentes']*100
    return tab.sort_values('acidentes', ascending=False), tab[tab['acidentes'] >= minimo].sort_values('pct_fatais', ascending=False)

if C_CAUSA:
    causas_volume, causas_gravidade = resumo_categoria_fatal(C_CAUSA, minimo=max(100, int(len(df)*0.0001)))
    print('Causas mais frequentes:')
    display(causas_volume.head(12).round(2))
    print('Causas com maior proporção de fatalidade (com volume mínimo):')
    display(causas_gravidade.head(12).round(2))

    plt.figure(figsize=(11,6))
    top = causas_volume.head(10).sort_values('acidentes')
    plt.barh(top.index, top['acidentes'])
    plt.title('Top 10 causas por volume de acidentes')
    plt.xlabel('Acidentes')
    plt.tight_layout()
    plt.show()

if C_TIPO:
    tipos_volume, tipos_gravidade = resumo_categoria_fatal(C_TIPO, minimo=max(100, int(len(df)*0.0001)))
    print('Tipos mais frequentes:')
    display(tipos_volume.head(12).round(2))
    print('Tipos com maior proporção de fatalidade (com volume mínimo):')
    display(tipos_gravidade.head(12).round(2))

    top = tipos_volume.head(12).index
    plot_tab = df[df[C_TIPO].isin(top)].groupby(C_TIPO)['fatal'].agg(['count','mean'])
    plot_tab['pct_fatais'] = plot_tab['mean']*100
    plot_tab = plot_tab.sort_values('pct_fatais')
    plt.figure(figsize=(10,6))
    plt.barh(plot_tab.index, plot_tab['pct_fatais'])
    plt.xlabel('% de acidentes fatais')
    plt.title('Proporção de fatalidade entre os 12 tipos mais frequentes')
    plt.tight_layout()
    plt.show()

    print('DISCUSSÃO: comparar apenas a quantidade de acidentes pode esconder tipos menos frequentes, porém proporcionalmente mais graves.')

## 12. Pergunta 5 — Clima e características da via

In [ ]:
for col, titulo in [(C_CLIMA, 'Condição meteorológica'), (C_PISTA, 'Tipo de pista'), (C_TRACADO, 'Traçado da via')]:
    if not col:
        continue
    tab = df.groupby(col)['fatal'].agg(acidentes='size', fatais='sum')
    tab['pct_fatais'] = tab['fatais']/tab['acidentes']*100
    tab = tab.sort_values('acidentes', ascending=False)
    print(f'\n### {titulo}')
    display(tab.head(15).round(2))

    top = tab.head(10).sort_values('acidentes')
    plt.figure(figsize=(10,5))
    plt.barh(top.index.astype(str), top['acidentes'])
    plt.title(f'Acidentes por {titulo.lower()} — top categorias')
    plt.xlabel('Acidentes')
    plt.tight_layout()
    plt.show()

    top_taxa = tab[tab['acidentes'] >= max(100, int(len(df)*0.0001))].sort_values('pct_fatais', ascending=False).head(10).sort_values('pct_fatais')
    plt.figure(figsize=(10,5))
    plt.barh(top_taxa.index.astype(str), top_taxa['pct_fatais'])
    plt.title(f'% de acidentes fatais por {titulo.lower()}')
    plt.xlabel('% fatais')
    plt.tight_layout()
    plt.show()

print('DISCUSSÃO: diferenças de proporção são associações descritivas. Sem controlar fluxo, velocidade, região e outras variáveis, não devemos afirmar causalidade.')

## 13. Princípio relacional — quantitativa x quantitativa

In [ ]:
if len(VARS_QUANT) >= 2:
    amostra_corr = df[VARS_QUANT].sample(min(300000, len(df)), random_state=RANDOM_STATE)
    corr_spearman = amostra_corr.corr(method='spearman')
    corr_pearson = amostra_corr.corr(method='pearson')

    fig, axes = plt.subplots(1, 2, figsize=(15,6))
    sns.heatmap(corr_pearson, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=axes[0])
    axes[0].set_title('Correlação de Pearson')
    sns.heatmap(corr_spearman, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=axes[1])
    axes[1].set_title('Correlação de Spearman')
    plt.tight_layout()
    plt.show()

    # Maior correlação Spearman fora da diagonal
    matriz = corr_spearman.abs().where(~np.eye(len(corr_spearman), dtype=bool))
    par = matriz.stack().idxmax()
    r = corr_spearman.loc[par[0], par[1]]
    print(f'DISCUSSÃO: maior |Spearman| observado entre {par[0]} e {par[1]}: r={r:.3f}.')
    print('Spearman é priorizado porque as contagens são assimétricas, discretas e apresentam valores extremos.')

In [ ]:
if C_PESSOAS and C_VEICULOS:
    amostra_scatter = df[[C_PESSOAS, C_VEICULOS, 'fatal']].dropna().sample(
        min(30000, df[[C_PESSOAS, C_VEICULOS]].dropna().shape[0]), random_state=RANDOM_STATE
    )
    plt.figure(figsize=(8,6))
    sns.scatterplot(data=amostra_scatter, x=C_VEICULOS, y=C_PESSOAS, hue='fatal', alpha=.35)
    plt.title('Veículos x pessoas envolvidas (amostra)')
    plt.tight_layout()
    plt.show()

    r_s, p_s = stats.spearmanr(amostra_scatter[C_VEICULOS], amostra_scatter[C_PESSOAS], nan_policy='omit')
    print(f'Spearman veículos x pessoas: r={r_s:.3f}, p={p_s:.3g}')

## 14. Quantitativa x qualitativa — comparação de grupos

In [ ]:
# Exemplo: número de feridos entre fases do dia. Kruskal-Wallis é adequado a dados não normais.
if C_FERIDOS and C_FASE:
    top_fases = df[C_FASE].value_counts().head(6).index
    amostra_kw = df[df[C_FASE].isin(top_fases)][[C_FASE, C_FERIDOS]].dropna().sample(min(120000, df[df[C_FASE].isin(top_fases)][[C_FASE, C_FERIDOS]].dropna().shape[0]), random_state=RANDOM_STATE)
    grupos = [g[C_FERIDOS].values for _, g in amostra_kw.groupby(C_FASE)]
    grupos = [g for g in grupos if len(g) > 0]

    if len(grupos) >= 2:
        stat_kw, p_kw = stats.kruskal(*grupos)
        print(f'Kruskal-Wallis — {C_FERIDOS} entre categorias de {C_FASE}: estatística={stat_kw:.3f}, p={p_kw:.4g}')
        if p_kw < 0.05:
            print('Resultado: há evidência de diferença na distribuição de feridos entre pelo menos duas fases do dia.')
        else:
            print('Resultado: não há evidência suficiente de diferença entre as fases do dia.')

        plt.figure(figsize=(10,6))
        amostra_box = df[df[C_FASE].isin(top_fases)].sample(min(80000, len(df[df[C_FASE].isin(top_fases)])), random_state=RANDOM_STATE)
        sns.boxplot(data=amostra_box, x=C_FASE, y=C_FERIDOS, showfliers=False, color=COR_PRINCIPAL)
        plt.xticks(rotation=25)
        plt.title(f'{C_FERIDOS} por {C_FASE} — boxplot sem extremos visuais')
        plt.tight_layout()
        plt.show()

## 15. Qualitativa x qualitativa — teste do Qui-Quadrado

In [ ]:
# Tipo de acidente x classificação de gravidade, usando os tipos mais frequentes para tabela legível.
if C_TIPO and C_CLASS:
    top_tipos = df[C_TIPO].value_counts().head(10).index
    sub = df[df[C_TIPO].isin(top_tipos)][[C_TIPO, C_CLASS]].dropna()
    contingencia = pd.crosstab(sub[C_TIPO], sub[C_CLASS])
    chi2, p_chi, gl, esperado = stats.chi2_contingency(contingencia)

    display(contingencia)
    print(f'Qui-Quadrado: chi2={chi2:.2f}, gl={gl}, p={p_chi:.3g}')
    print('Interpretação:', 'há associação estatisticamente significativa.' if p_chi < 0.05 else 'não há evidência de associação.')

    prop = pd.crosstab(sub[C_TIPO], sub[C_CLASS], normalize='index') * 100
    plt.figure(figsize=(10,7))
    sns.heatmap(prop, annot=True, fmt='.1f', cmap='magma')
    plt.title('Distribuição percentual da classificação dentro de cada tipo de acidente')
    plt.xlabel('Classificação')
    plt.ylabel('Tipo de acidente')
    plt.tight_layout()
    plt.show()

    print('DISCUSSÃO: significância estatística não informa causalidade; ela apenas indica que a distribuição de gravidade não é independente do tipo de acidente.')

## 16. Análise multivariada — perfil de gravidade por contexto

In [ ]:
# Combina fase do dia e tipo de pista para observar como a proporção de fatalidade varia em dois contextos simultaneamente.
if C_FASE and C_PISTA:
    multi = df.groupby([C_FASE, C_PISTA])['fatal'].agg(['count','mean']).reset_index()
    multi = multi[multi['count'] >= max(100, int(len(df)*0.00005))]
    pivot_multi = multi.pivot(index=C_FASE, columns=C_PISTA, values='mean') * 100
    display(pivot_multi.round(2))

    plt.figure(figsize=(9,5))
    sns.heatmap(pivot_multi, annot=True, fmt='.2f', cmap='magma')
    plt.title('% de acidentes fatais — fase do dia x tipo de pista')
    plt.tight_layout()
    plt.show()

    print('DISCUSSÃO: a combinação de variáveis pode revelar padrões que não aparecem olhando cada coluna isoladamente. Ainda assim, a análise é associativa e não causal.')

## 17. Fairness — aplicabilidade neste dataset

A aula apresenta avaliação de *fairness* quando existem atributos sensíveis, como sexo e faixa etária. Neste conjunto escolhido, a unidade de análise é o **acidente agregado**, e não uma pessoa. A versão utilizada nesta EDA não possui atributos demográficos individuais adequados para uma avaliação séria de fairness.

Por isso, **não seria correto inventar uma análise de justiça algorítmica** apenas para repetir o exemplo da aula. Caso fossem usados dados da PRF agrupados por pessoa, com sexo/idade e desfecho individual, então essa análise poderia ser realizada com os devidos cuidados éticos e de representatividade.

## 18. Checagem final de qualidade após o tratamento conservador

In [ ]:
print('Linhas brutas:', f'{linhas_brutas:,}')
print('Linhas atuais:', f'{len(df):,}')
print('Duplicatas exatas atuais:', f'{df.duplicated().sum():,}')
print('Período:', df['ano'].min(), 'a', df['ano'].max())

# Não removemos outliers nem descartamos linhas globalmente; portanto, a distribuição original é preservada.
print('\nDecisão de tratamento:')
print('- Ausentes: tratados por análise (dropna apenas quando necessário).')
print('- Outliers: mantidos, pois podem representar acidentes raros e reais.')
print('- Tipos: conversões pontuais para data, contagens, BR e KM.')

## 19. Síntese — principais conclusões

In [ ]:
conclusoes = []

# 1. temporal
if df['ano'].notna().any():
    temp = df.groupby('ano').size()
    pico_ano = int(temp.idxmax())
    pico_n = int(temp.max())
    conclusoes.append(f'1. O maior volume anual de acidentes ocorreu em {pico_ano}, com {pico_n:,} registros na base.')

# 2. UF
if C_UF:
    uf_top = df[C_UF].value_counts().index[0]
    uf_pct = df[C_UF].value_counts(normalize=True).iloc[0]*100
    conclusoes.append(f'2. {uf_top} é a UF com maior participação no total de acidentes ({uf_pct:.2f}%), mas isso mede concentração de registros, não risco por exposição.')

# 3. tipo
if C_TIPO:
    tab = df.groupby(C_TIPO)['fatal'].agg(['count','mean'])
    lim = max(100, int(len(df)*0.0001))
    tab = tab[tab['count'] >= lim]
    tipo_grave = tab['mean'].idxmax()
    pct = tab.loc[tipo_grave,'mean']*100
    conclusoes.append(f'3. Entre tipos com volume mínimo relevante, "{tipo_grave}" apresenta uma das maiores proporções de acidentes fatais ({pct:.2f}%).')

# 4. pista/fase
if C_FASE:
    tf = df.groupby(C_FASE)['fatal'].mean().sort_values(ascending=False)
    conclusoes.append(f'4. A fase do dia com maior proporção de acidentes fatais é "{tf.index[0]}" ({tf.iloc[0]*100:.2f}%), mostrando que volume e gravidade devem ser analisados separadamente.')

# 5. missingness/outlier
conclusoes.append('5. O tratamento foi conservador: ausências foram tratadas conforme a análise e valores extremos foram mantidos quando plausíveis, evitando apagar acidentes raros e potencialmente relevantes.')

print('\n'.join(conclusoes[:5]))

### Interpretação geral

A EDA mostra que uma boa leitura deste conjunto exige separar **frequência** de **gravidade**. Estados, rodovias ou condições com muitos registros não são automaticamente os mais perigosos, porque a base não contém denominadores de exposição. Da mesma forma, tipos de acidente menos comuns podem ter uma proporção de fatalidade muito maior.

As análises estatísticas foram escolhidas de acordo com a natureza dos dados: contagens discretas, assimétricas e com extremos reais. Por isso foram priorizadas medidas robustas e métodos não paramétricos.

## 20. Declaração de Uso de IA

**Ferramenta utilizada:** ChatGPT (OpenAI).

**Para que foi utilizada:** apoio na organização da EDA, sugestão de perguntas, revisão do código Python, escolha de gráficos/testes estatísticos e redação inicial das discussões.

**O que foi aceito/modificado/rejeitado:** a estrutura e o código foram aceitos após validação no próprio notebook. Foram modificadas interpretações para evitar afirmar causalidade onde os dados mostram apenas associação. Foram rejeitadas conclusões que tratassem quantidade de acidentes como “risco” sem considerar exposição ao tráfego, extensão das rodovias ou quilômetros percorridos.

## Referências

- Kaggle. **Acidentes em Rodovias Federais (2007–2020)**. https://www.kaggle.com/datasets/marcosfnlr/acidentes-em-rodovias-federais-20072020/data
- Polícia Rodoviária Federal. **Dicionário de variáveis — acidentes agrupados por ocorrência**. https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dicionario-dados-acidentes-2016
- SciPy. Testes estatísticos `shapiro`, `kruskal`, `chi2_contingency`.
- Pandas / Matplotlib / Seaborn — manipulação e visualização dos dados.

## Exportação dos dataframes para o Streamlit (Aulas 21–23)

O app (`app.py`) não roda dentro do notebook, então salvamos aqui:
- **original** → `data/raw/dataset.csv` (cópia da base do Kaggle; **não** vai para o Git por ser grande — está no `.gitignore`);
- **tratado** → `data/clean/dataset.csv` e `data/clean/dataset.parquet`.

Tratamento aplicado: `(null)` vira valor ausente, a data (fim de `id_unico`) vira a coluna `ano`, ficam só as colunas usadas no app e `feridos` é removida porque é sempre `feridos_leves + feridos_graves` (o app recalcula).
O CSV tratado tem ~104 MB e passa do limite de 100 MB por arquivo do GitHub; por isso o app e o deploy usam o **Parquet** (~7 MB), formato sugerido na Aula 23 para economizar memória.

In [ ]:
import shutil
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
pasta_raw, pasta_clean = RAIZ / 'data' / 'raw', RAIZ / 'data' / 'clean'
pasta_raw.mkdir(parents=True, exist_ok=True)
pasta_clean.mkdir(parents=True, exist_ok=True)

# 1) Dataframe original: cópia fiel do arquivo baixado
shutil.copy(csv_path, pasta_raw / 'dataset.csv')

# 2) Dataframe tratado para o app
QUANT_APP = ['pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos', 'ignorados', 'veiculos']
CATEG_APP = ['uf', 'classificacao_acidente', 'fase_dia', 'condicao_metereologica']
bruto = pd.read_csv(csv_path, low_memory=False, na_values=['(null)'],
                    usecols=lambda c: c in ['id_unico', 'feridos'] + QUANT_APP + CATEG_APP)
tratado = pd.DataFrame({'ano': pd.to_datetime(bruto['id_unico'].astype(str).str[-10:],
                                             errors='coerce', format='%Y-%m-%d').dt.year.astype('Int16')})
for c in CATEG_APP:
    tratado[c] = bruto[c].astype('string').str.strip()
for c in QUANT_APP:
    tratado[c] = pd.to_numeric(bruto[c], errors='coerce').astype('Int32')
assert (bruto['feridos'] == bruto['feridos_leves'] + bruto['feridos_graves']).all()

tratado.to_csv(pasta_clean / 'dataset.csv', index=False)
compacto = tratado.copy()
for c in CATEG_APP:
    compacto[c] = compacto[c].astype('category')
for c in ['ano'] + QUANT_APP:
    compacto[c] = pd.to_numeric(compacto[c], downcast='integer')
compacto.to_parquet(pasta_clean / 'dataset.parquet', index=False)

print('Original :', pasta_raw / 'dataset.csv', bruto.shape)
print('Tratado  :', pasta_clean / 'dataset.parquet', tratado.shape)
display(tratado.head())